In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MAX_JOBS"] = "16"
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [2]:
import os
import argparse
import json
import numpy as np
import math
from tqdm import tqdm
from PIL import Image
from IPython.display import clear_output
import torch
import torch.distributed as dist
from diffusers.models import AutoencoderKL
import torch_fidelity
import matplotlib.pyplot as plt

from sit import SiT_models
from meanflow_sampler import meanflow_sampler

/home/iasudakov/miniconda3/envs/DiT/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from munch import Munch

args = Munch(
    # seed
    global_seed=0,
    
    # logging/saving
    ckpt="/home/iasudakov/MeanFlow/ImageNet256/sit_l_2_meanflow_ema.pt",
    sample_dir="samples",
    
    # model
    model="SiT-L/2",
    num_classes=1000,
    encoder_depth=8,
    resolution=256,
    
    # sampling
    per_proc_batch_size=128,
    num_fid_samples=50_000,
    num_steps=1,
    cfg_scale=1.0,
    
    # Evaluation metrics
    compute_metrics=False,
    fid_statistics_file=""
)

In [6]:
rank = 0
device = 0
seed = args.global_seed + rank
torch.manual_seed(seed)
torch.cuda.set_device(device)

In [7]:
# Load model:
block_kwargs = {"fused_attn": False, "qk_norm": False}
latent_size = args.resolution // 8
model = SiT_models[args.model](
    input_size=latent_size,
    num_classes=args.num_classes,
    use_cfg = True,
    **block_kwargs,
).to(device)

# Load checkpoint
state_dict = torch.load(args.ckpt, map_location=f'cuda:{device}')
model.load_state_dict(state_dict)
model.eval()
None

In [8]:
vae = AutoencoderKL.from_pretrained(f"stabilityai/sd-vae-ft-ema").to(device)
assert args.cfg_scale >= 1.0, "In almost all cases, cfg_scale should be >= 1.0"

/home/iasudakov/miniconda3/envs/DiT/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [9]:
# Create folder to save samples:
model_string_name = args.model.replace("/", "-")
ckpt_string_name = os.path.basename(args.ckpt).replace(".pt", "") if args.ckpt else "pretrained"
folder_name = f"meanflow-{model_string_name}-{ckpt_string_name}-size-{args.resolution}-" \
                f"cfg-{args.cfg_scale}-steps-{args.num_steps}-seed-{args.global_seed}"
eval_fid_dir = f"{args.sample_dir}/{folder_name}"
img_folder = eval_fid_dir+'/img_dir'
if rank == 0:
    os.makedirs(eval_fid_dir, exist_ok=True)            # saving FID results.
    os.makedirs(img_folder, exist_ok=True)              # saving images
    print(f"Saving .png samples at {eval_fid_dir}")

n = args.per_proc_batch_size
global_batch_size = n
total_samples = int(math.ceil(args.num_fid_samples / global_batch_size) * global_batch_size)
if rank == 0:
    print(f"Total number of images that will be sampled: {total_samples}")
    print(f"SiT Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Using {args.num_steps}-step sampling")

samples_needed_this_gpu = int(total_samples)
assert samples_needed_this_gpu % n == 0, "samples_needed_this_gpu must be divisible by the per-GPU batch size"
iterations = int(samples_needed_this_gpu // n)
pbar = range(iterations)
pbar = tqdm(pbar) if rank == 0 else pbar
total = 0

Saving .png samples at samples/meanflow-SiT-L-2-sit_l_2_meanflow_ema-size-256-cfg-1.0-steps-1-seed-0
Total number of images that will be sampled: 50048
SiT Parameters: 459,399,184
Using 1-step sampling


  0%|          | 0/391 [00:00<?, ?it/s]

In [14]:
import PIL
import torchvision.transforms as transforms
from tqdm import tqdm

data_path = '/home/iasudakov/GenClass_local/imagenet_data/imagenet_val'

In [16]:
def center_crop_arr(pil_image, image_size):
    while min(*pil_image.size) >= 2 * image_size:
        pil_image = pil_image.resize(
            tuple(x // 2 for x in pil_image.size), resample=Image.BOX
        )

    scale = image_size / min(*pil_image.size)
    pil_image = pil_image.resize(
        tuple(round(x * scale) for x in pil_image.size), resample=Image.BICUBIC
    )

    arr = np.array(pil_image)
    crop_y = (arr.shape[0] - image_size) // 2
    crop_x = (arr.shape[1] - image_size) // 2
    return Image.fromarray(arr[crop_y: crop_y + image_size, crop_x: crop_x + image_size]), arr.shape[0], arr.shape[1], crop_y, crop_x


In [ ]:
n_class = 1000
batch_size = 200
step_size = 100

true = 0
cnt = 0

inds = torch.arange(n_class)

YS = torch.tensor(inds).to(device)
accs = []
incorrect = []


generator = torch.Generator()
generator.manual_seed(42)

for ind in inds:
    log_probs = torch.zeros(n_class).to(device)
    file_name = f'{data_path}/{ind}_{0}.JPEG'

    img = PIL.Image.open(file_name).convert('RGB')
    transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
            ])
    
    img = center_crop_arr(img, args.resolution)[0]

    img = transform(img).to(device).unsqueeze(0)

    ts = torch.arange(step_size//2, 1000, step_size).to(device)/1000
    with torch.no_grad():
        latents = vae.encode(img).latent_dist.sample().mul_(0.18215)    

        for t in ts:
            noise = torch.randn(latents.shape, generator=generator).to(device)
            noised_latents = (1-t)*latents + t*noise
            noised_latents = noised_latents.tile((batch_size, 1, 1, 1))
            t = torch.full((batch_size,), t, device=device)
            r = torch.full((batch_size,), 0.0, device=device)

            for batch_ind in range(n_class//batch_size):
                ys = YS[batch_ind*batch_size:(batch_ind+1)*batch_size]

                u = model(noised_latents, r, t, y=ys)

                v_target = noise - latents


                x_0_pred = noised_latents - t[0] * u
                eps_pred = (noised_latents - (1-t[0])*x_0_pred)/t[0]

                loss = ((eps_pred - noise)**2).mean(dim = (1, 2, 3))
                # loss = ((x_0_pred - latents)**2).mean(dim = (1, 2, 3))
                # loss = ((u - v_target)**2).mean(dim=(1, 2, 3))

                log_probs[batch_ind*batch_size:(batch_ind+1)*batch_size] -= loss

    cnt += 1
    if ind == YS[log_probs.argmax()]:
        true +=1

    print(ind.item(), true/cnt)

print('acc =', true/cnt)

/var/tmp/ipykernel_4171775/151442519.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  YS = torch.tensor(inds).to(device)


0 0.0
1 0.0
2 0.0
3 0.0


In [22]:
# eps pred acc = 0.5
# x pred acc = 0.1
# v pred acc = -